In [ ]:
import os
import bz2
import xml.etree.ElementTree as ET
import re
from glob import glob

In [ ]:
def extract_wikipedia_text(bz2_path, output_path):
    """Extract plain text from a Wikipedia dump (pages-articles)."""
    with bz2.open(bz2_path, "rt", encoding="utf-8", errors="ignore") as f, open(output_path, "w", encoding="utf-8") as out:
        buffer = []
        for line in f:
            if line.startswith("<text "):
                buffer = [line]
            elif line.startswith("</text>"):
                buffer.append(line)
                # join, strip tags
                txt = "".join(buffer)
                # naive remove XML tags
                clean = re.sub(r"<[^>]+>", "", txt)
                out.write(clean + "\n")
            elif buffer:
                buffer.append(line)

def read_transcript_files(folder, extension_list=("txt","tsv","linear-orthographic")):
    """Yield sentences from transcripts folder (KIParla)."""
    for ext in extension_list:
        for path in glob(os.path.join(folder, "**", f"*.{ext}"), recursive=True):
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    yield line

def normalize_sentence(s: str) -> str:
    s = s.strip()
    # optionally lower, remove odd chars
    return s

def build_combined_corpus(wiki_bz2=None, kip_folder=None, parla_to_folder=None, other_folders=[], out_path=None):
    assert(out_path is not None)
    with open(out_path, "w", encoding="utf-8") as outf:
        # Wikipedia first
        if wiki_bz2:
            print("Extracting Wikipedia …")
            extract_wikipedia_text(wiki_bz2, "wiki_plain.txt")
            with open("wiki_plain.txt", "r", encoding="utf-8", errors="ignore") as wfin:
                for ln in wfin:
                    ln2 = normalize_sentence(ln)
                    if ln2:
                        outf.write(ln2 + "\n")

        # KIParla KIP transcripts
        if kip_folder:
            print("Including KIP transcripts …")
            for sent in read_transcript_files(kip_folder):
                sent2 = normalize_sentence(sent)
                if sent2:
                    outf.write(sent2 + "\n")

        # ParlaTO
        if parla_to_folder:
            print("Including ParlaTO transcripts …")
            for sent in read_transcript_files(parla_to_folder):
                sent2 = normalize_sentence(sent)
                if sent2:
                    outf.write(sent2 + "\n")

        # Other folders (e.g. Leipzig, news text dumps)
        if other_folders:
            for folder in other_folders:
                for path in glob(os.path.join(folder, "**", "*.txt"), recursive=True):
                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        for line in f:
                            ln2 = normalize_sentence(line)
                            if ln2:
                                outf.write(ln2 + "\n")

In [ ]:
# wiki_bz2 = "itwiki-latest-pages-articles.xml.bz2"
wiki_bz2 = None  # Set to None if Wikipedia dump is not available
kip_folder = "KIP_transcripts"
parla_to_folder = "ParlaTO_transcripts"
# other_folders = ["Leipzig_it", "news_texts"]
other_folders = []
out_path = "combined_italian_corpus.txt"
build_combined_corpus(
    wiki_bz2=wiki_bz2, 
    kip_folder=kip_folder, 
    parla_to_folder=parla_to_folder, 
    other_folders=other_folders, 
    out_path='combined_italian_corpus.txt'
)
print("Done, written to", out_path)